# Решение

Сначала отсеиваю заведомо неподходящие объявления по структурным признакам, затем ранжирую оставшихся кандидатов с помощью BGE-M3 по cosine similarity.

In [ ]:
from pathlib import Path
import html
import re
import gc

import numpy as np
import pandas as pd
import torch
import faiss

from tqdm.auto import tqdm
from FlagEmbedding import BGEM3FlagModel

In [ ]:
top_k = 50 # по условию нужно вернуть не больше 50 объявлений

# Для объявления использую до 1024 токенов (костыль)
# Запросы короткие, поэтому для них оставляю полное окно модели.

item_max_length = 1024
query_max_length = 8192

cache_dir = Path('cache/final')
cache_dir.mkdir(parents=True, exist_ok=True)

`train.parquet` использую не для обучения BGE, а для восстановления связи между поисковой категорией и категориями объявлений.

Предсказания строю для запросов из `benchmark_queries.parquet` по корпусу `benchmark_items.parquet`.

`query_id` и `item_id` сразу сохраняю строками, чтобы не потерять формат идентификаторов.

In [ ]:
train = pd.read_parquet('train.parquet')
benchmark_items = pd.read_parquet('benchmark_items.parquet')
benchmark_queries = pd.read_parquet('benchmark_queries.parquet')

# ID храню строками
train['item_id'] = train['item_id'].astype(str)
benchmark_items['item_id'] = benchmark_items['item_id'].astype(str)
benchmark_queries['query_id'] = benchmark_queries['query_id'].astype(str)

print('train:', train.shape)
print('benchmark_items:', benchmark_items.shape)
print('benchmark_queries:', benchmark_queries.shape)

train: (497673, 19)
benchmark_items: (189212, 14)
benchmark_queries: (2452, 6)


Для текста убираю HTML-сущности и лишние пробелы.

Для текстовых фильтров дополнительно привожу строку к нижнему регистру, заменяю `ё → е` и оставляю только буквы и цифры.

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ''

    x = html.unescape(str(x))
    return re.sub(r'\s+', ' ', x).strip()


def clean_query(x):
    if pd.isna(x):
        return ''

    x = str(x).lower().replace('ё', 'е')

    return ' '.join(
        re.findall(r'[a-zа-я0-9]+', x)
    )

## Предварительная фильтрация кандидатов

Главный прирост качества дала фильтрация до векторного поиска.

По положительным взаимодействиям из train для каждой `search_category` собираю допустимые `item_category_id` и `item_microcat_id`.

Для каждого запроса затем оставляю только объявления, которые проходят:

1. категорию и микрокатегорию
2. точное совпадение `location_id`
3. дополнительные параметры из `search_infm_params_text`

Для параметров требую, чтобы хотя бы один токен длиной от 3 символов встречался в `item_infm_params_text`.

`search_is_delivery_search` не использую как жёсткий фильтр

In [ ]:
cat_map = (
    train
    .dropna(subset=['search_category'])
    .groupby('search_category')['item_category_id']
    .agg(lambda x: set(x.dropna()))
    .to_dict()
)

microcat_map = (
    train
    .dropna(subset=['search_category'])
    .groupby('search_category')['item_microcat_id']
    .agg(lambda x: set(x.dropna()))
    .to_dict()
)

In [ ]:
# Перевожу признаки benchmark-объявлений в numpy
# дальше по ним много раз строятся булевы маски

item_category = benchmark_items['item_category_id'].to_numpy()
item_microcat = benchmark_items['item_microcat_id'].to_numpy()
item_location = benchmark_items['item_location_id'].to_numpy()

item_params = (
    benchmark_items['item_infm_params_text']
    .fillna('')
    .astype(str)
    .str.lower()
    .str.replace('ё', 'е')
    .to_numpy(dtype=str)
)

In [ ]:
# Оставляю объявления, прошедшие структурные фильтры

def allowed_items_for_query(qrow):

    mask = np.ones(
        len(benchmark_items),
        dtype=bool
    )

    # Категория и микрокатегория
    
    search_cat = qrow['search_category']

    if not pd.isna(search_cat):

        allowed_cat = cat_map.get(search_cat)
        allowed_micro = microcat_map.get(search_cat)

        if allowed_cat:
            mask &= np.isin(
                item_category,
                list(allowed_cat)
            )

        if allowed_micro:
            mask &= np.isin(
                item_microcat,
                list(allowed_micro)
            )

    # Точная локация
    
    search_loc = qrow['search_location_id']

    if not pd.isna(search_loc):

        mask &= (
            item_location == search_loc
        )

    # Дополнительные параметры запроса

    params = qrow['search_infm_params_text']

    if not pd.isna(params):

        tokens = [
            token
            for token in clean_query(params).split()
            if len(token) >= 3
        ]

        if tokens:

            candidate_indices = np.flatnonzero(mask)

            param_mask = np.zeros(
                len(candidate_indices),
                dtype=bool
            )

            texts = item_params[candidate_indices]

            for token in tokens:

                param_mask |= (
                    np.char.find(
                        texts,
                        token
                    ) >= 0
                )

            mask[:] = False

            mask[
                candidate_indices[param_mask]
            ] = True

    return np.flatnonzero(mask)

## Фильтрация benchmark

Применяю фильтры ко всем benchmark-запросам.

Для каждого запроса сохраняю набор допустимых объявлений. Одновременно собираю объединение всех объявлений, которые прошли фильтры хотя бы для одного запроса.

Это позволяет дальше посчитать BGE-эмбеддинг каждого нужного объявления только один раз, а не заново для каждого запроса.

In [ ]:
allowed_by_query = [] # кандидаты для каждого запроса

needed_items = set() # все уникальные объявления, которые понадобятся BGE

for _, qrow in tqdm(
    benchmark_queries.iterrows(),
    total=len(benchmark_queries),
    desc='Filtering'
):

    allowed = allowed_items_for_query(qrow)

    allowed_by_query.append(allowed)

    needed_items.update(
        allowed.tolist()
    )


needed_items = np.asarray(
    sorted(needed_items),
    dtype=np.int32
)

## Векторные представления BGE-M3

Запрос представляется как:

`search_query + search_infm_params_text`

Объявление:

`item_title_raw + item_infm_params_text + item_description_raw`

Использую готовую `BAAI/bge-m3` без дообучения. Для объявлений ограничиваю текст 1024 токенами (костыль:)).

Эмбеддинги сохраняю в cache, чтобы при повторном запуске не считать их заново.

In [ ]:
# Текст, который подаётся в BGE

def make_item_text(row):

    return (
        clean_text(row['item_title_raw'])
        + '\n'
        + clean_text(row['item_infm_params_text'])
        + '\n'
        + clean_text(row['item_description_raw'])
    )


def make_query_text(row):

    return (
        clean_text(row['search_query'])
        + '\n'
        + clean_text(row['search_infm_params_text'])
    )

In [ ]:
# Пути для кеширования рассчитанных эмбеддингов
item_rows_path = (
    cache_dir / 'benchmark_item_rows_1024.npy'
)

item_emb_path = (
    cache_dir / 'benchmark_item_bge_1024.npy'
)

query_emb_path = (
    cache_dir / 'benchmark_query_bge.npy'
)

In [ ]:
# Использую кеш только если он рассчитан для того же набора объявлений.
use_cached_items = False

if (
    item_rows_path.exists()
    and item_emb_path.exists()
):

    saved_rows = np.load(
        item_rows_path
    )

    if np.array_equal(
        saved_rows,
        needed_items
    ):
        use_cached_items = True


need_model = (
    not use_cached_items
    or not query_emb_path.exists()
)
# Считаю эмбеддинги только для объявлений, которые остались хотя бы у одного запроса после фильтрации.

if need_model:
    model = BGEM3FlagModel(
        'BAAI/bge-m3',
        use_fp16=torch.cuda.is_available(),
        devices='cuda:0' if torch.cuda.is_available() else 'cpu',
    )

if use_cached_items:
    item_vecs = np.load(item_emb_path)

else:
    needed_item_texts = [
        make_item_text(benchmark_items.iloc[item_idx])
        for item_idx in needed_items
    ]

    item_vecs = model.encode_corpus(
        needed_item_texts,
        batch_size=8,
        max_length=item_max_length,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )['dense_vecs'].astype('float32')

    np.save(item_emb_path, item_vecs)
    np.save(item_rows_path, needed_items)

In [ ]:
# Один эмбеддинг на каждый запрос бенчмарка.

if query_emb_path.exists():
    query_vecs = np.load(query_emb_path)

else:
    benchmark_query_texts = [
        make_query_text(row)
        for _, row in benchmark_queries.iterrows()
    ]

    query_vecs = model.encode_queries(
        benchmark_query_texts,
        batch_size=64,
        max_length=query_max_length,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )['dense_vecs'].astype('float32')

    np.save(query_emb_path, query_vecs)

## Подготовка к ранжированию

Нормализую BGE-векторы по L2. После этого скалярное произведение двух векторов соответствует cosine similarity.

Также сохраняю соответствие между индексом объявления в исходном бенчмарке и строкой его рассчитанного эмбеддинга.

In [ ]:
if need_model:

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
faiss.normalize_L2(
    item_vecs
)

faiss.normalize_L2(
    query_vecs
)

In [ ]:
# Индекс объявления в benchmark_items -> индекс в item_vecs.

global_to_local = {
    int(global_idx): local_idx
    for local_idx, global_idx
    in enumerate(needed_items)
}

## Ранжирование

Для каждого запроса беру только объявления, которые прошли фильтры.

Считаю cosine similarity между эмбеддингом запроса и эмбеддингами этих кандидатов и выбираю 50 объявлений с максимальным скором.

Таким образом, структурные признаки отвечают за отбор допустимых кандидатов, а BGE - за семантическое ранжирование внутри этого набора.

In [ ]:
# Считаю cosine similarity только внутри отфильтрованного набора.

answers = []

for qi, allowed in enumerate(
    tqdm(
        allowed_by_query,
        desc='Ranking'
    )
):

    # если фильтры вообще всё вырезали
    if len(allowed) == 0:

        answers.append('')
        continue


    local_rows = np.asarray(
        [
            global_to_local[int(item_idx)]
            for item_idx in allowed
        ],
        dtype=np.int32
    )


    # cosine similarity
    scores = (
        item_vecs[local_rows]
        @ query_vecs[qi]
    )


    k = min(
        top_k,
        len(allowed)
    )


    if len(allowed) > k:

        top = np.argpartition(
            -scores,
            k - 1
        )[:k]

        top = top[
            np.argsort(
                scores[top]
            )[::-1]
        ]

    else:

        top = np.argsort(
            scores
        )[::-1]


    global_rows = allowed[top]


    predicted_ids = (
        benchmark_items
        .iloc[global_rows]['item_id']
        .astype(str)
        .tolist()
    )


    # защита от дублей
    predicted_ids = list(
        dict.fromkeys(predicted_ids)
    )[:top_k]


    answers.append(
        ' '.join(predicted_ids)
    )

In [ ]:
answer = pd.DataFrame({
    'query_id':
        benchmark_queries['query_id'].astype(str),

    'answer':
        answers,
})

answer.to_csv(
    'answer.csv',
    index=False
)